# Get Product Data from a Public API

**Topic:** GET request, parameters and JSON response  
**Level:** Beginner  
**Time:** 35–40 minutes

## Scenario

An online shop needs current product information. We will call the
DummyJSON public API, convert its JSON response into a table and save it
as a CSV file.

No API key is required. If the internet is unavailable, the notebook
uses products_fallback.csv.

## Step 1: Create the fallback CSV dataset

In [2]:
from pathlib import Path

# This CSV is used when the public API is unavailable.
Path("products_fallback.csv").write_text(
    'product_id,title,brand,category,api_price,rating,stock\n1,Essence Mascara Lash Princess,Essence,beauty,9.99,4.94,5\n2,Eyeshadow Palette with Mirror,Glamour Beauty,beauty,19.99,4.28,44\n3,Powder Canister,Velvet Touch,beauty,14.99,3.82,59\n4,Red Lipstick,Chic Cosmetics,beauty,12.99,4.51,68\n5,Red Nail Polish,Nail Couture,beauty,8.99,3.91,71\n6,Calvin Klein CK One,Calvin Klein,fragrances,49.99,4.85,17\n7,Chanel Coco Noir Eau De,Chanel,fragrances,129.99,4.26,41\n8,Dior Jadore,Dior,fragrances,89.99,4.31,91\n9,Dolce Shine Eau de,Dolce and Gabbana,fragrances,69.99,3.77,3\n10,Gucci Bloom Eau de,Gucci,fragrances,79.99,4.69,93\n',
    encoding="utf-8"
)

print("Created: products_fallback.csv")

Created: products_fallback.csv


## Step 2: Import libraries and define the API

- requests sends the HTTP request.
- pandas works with the returned records as a table.

In [3]:
import pandas as pd
import requests

api_url = "https://dummyjson.com/products"
parameters = {"limit": 10}

print("API URL:", api_url)
print("Parameters:", parameters)

API URL: https://dummyjson.com/products
Parameters: {'limit': 10}


## Step 3: Send a GET request

A timeout prevents the program from waiting forever. The except block
loads the fallback CSV when the request fails.

In [4]:
try:
    response = requests.get(
        api_url,
        params=parameters,
        timeout=10
    )
    response.raise_for_status()

    api_data = response.json()
    products = pd.DataFrame(api_data["products"])

    products = products[
        ["id", "title", "brand", "category", "price", "rating", "stock"]
    ]
    products = products.rename(
        columns={"id": "product_id", "price": "api_price"}
    )

    data_source = "Live API"

except (requests.RequestException, ValueError) as error:
    products = pd.read_csv("products_fallback.csv")
    data_source = "Fallback CSV"
    print("The API was unavailable:", type(error).__name__)

print("Data source:", data_source)
print(products.head().to_string(index=False))

Data source: Live API
 product_id                         title          brand category  api_price  rating  stock
          1 Essence Mascara Lash Princess        Essence   beauty       9.99    2.56     99
          2 Eyeshadow Palette with Mirror Glamour Beauty   beauty      19.99    2.86     34
          3               Powder Canister   Velvet Touch   beauty      14.99    4.64     89
          4                  Red Lipstick Chic Cosmetics   beauty      12.99    4.36     91
          5               Red Nail Polish   Nail Couture   beauty       8.99    4.32     79


## Step 4: Filter useful products

Find products with a rating of at least 4.0 and stock greater than zero.

In [5]:
recommended_products = products[
    (products["rating"] >= 4.0)
    & (products["stock"] > 0)
]

recommended_products = recommended_products.sort_values(
    "rating",
    ascending=False
)

print(
    recommended_products[
        ["product_id", "title", "rating", "stock"]
    ].to_string(index=False)
)

 product_id                   title  rating  stock
          3         Powder Canister    4.64     89
          6     Calvin Klein CK One    4.37     29
          4            Red Lipstick    4.36     91
          5         Red Nail Polish    4.32     79
          7 Chanel Coco Noir Eau De    4.26     58


## Step 5: Create a simple category summary

In [6]:
category_summary = (
    products.groupby("category")
    .agg(
        number_of_products=("product_id", "count"),
        average_price=("api_price", "mean"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

category_summary["average_price"] = (
    category_summary["average_price"].round(2)
)
category_summary["average_rating"] = (
    category_summary["average_rating"].round(2)
)

print(category_summary.to_string(index=False))

  category  number_of_products  average_price  average_rating
    beauty                   5          13.39            3.75
fragrances                   5          83.99            3.83


## Step 6: Save API data as CSV

In [7]:
products.to_csv("api_products.csv", index=False)
recommended_products.to_csv(
    "recommended_products.csv",
    index=False
)

print("Created: api_products.csv")
print("Created: recommended_products.csv")

Created: api_products.csv
Created: recommended_products.csv


## Student task

1. Change the API limit from 10 to 5.
2. Display products with stock below 20.
3. Sort products from highest to lowest price.

## Expected learning

You can now send a GET request, read JSON, convert it to a DataFrame and
save API data as CSV.

## Student task

1. Change the API limit from 10 to 5.

In [8]:
parameters = {"limit": 5}

2. Display products with stock below 20.

In [9]:
low_stock_products = products[products["stock"] < 20]

print(low_stock_products.to_string(index=False))

 product_id              title           brand   category  api_price  rating  stock
          9 Dolce Shine Eau de Dolce & Gabbana fragrances      69.99    3.96      4


3. Sort products from highest to lowest price.

In [10]:
sorted_products = products.sort_values(
    "api_price",
    ascending=False
)

print(sorted_products.to_string(index=False))

 product_id                         title           brand   category  api_price  rating  stock
          7       Chanel Coco Noir Eau De          Chanel fragrances     129.99    4.26     58
          8                  Dior J'adore            Dior fragrances      89.99    3.80     98
         10            Gucci Bloom Eau de           Gucci fragrances      79.99    2.74     91
          9            Dolce Shine Eau de Dolce & Gabbana fragrances      69.99    3.96      4
          6           Calvin Klein CK One    Calvin Klein fragrances      49.99    4.37     29
          2 Eyeshadow Palette with Mirror  Glamour Beauty     beauty      19.99    2.86     34
          3               Powder Canister    Velvet Touch     beauty      14.99    4.64     89
          4                  Red Lipstick  Chic Cosmetics     beauty      12.99    4.36     91
          1 Essence Mascara Lash Princess         Essence     beauty       9.99    2.56     99
          5               Red Nail Polish    Nail 

# conclusion

Conclusion: This hands-on activity helped us understand how to retrieve product data from a public API using a GET request and process the JSON response using Pandas. We learned how to use API parameters, filter products based on stock availability, sort products according to price, and work with the resulting data in a DataFrame. This demonstrates how Python can be used to collect and analyze real-time data from public APIs.